Autoencoder 自编码器：从重建原理到去噪、异常检测与 VAE

自编码器（Autoencoder，AE）是一类以“重建输入”为训练目标的神经网络。给定输入 `x`，模型先将它压缩成潜在表示 `z`，再从 `z` 重建出 `x_hat`：

```text
x (输入) -> Encoder 编码器 -> z (潜在向量) -> Decoder 解码器 -> x_hat (重建结果)
```
Autoencoder 的核心思想是：将数据压缩成低维特征，再通过解码器重建原始数据。

结构：
原始输入 ➡️ Encoder（编码器） ➡️ Latent Space（潜在空间） ➡️ Decoder（解码器） ➡️ 重建数据

例如：
784 维图像 ➡️ 256 维 ➡️ 64 维 ➡️ 16 维（压缩特征） ➡️ 64 维 ➡️ 256 维 ➡️ 784 维（重建图像）

核心知识：
- Encoder / Decoder
- Latent Representation
- Reconstruction Loss
- MSELoss
- 降维与特征提取
经典应用：
- 图像降噪
- 异常检测
- 数据压缩
- 特征学习

推荐项目：使用 Autoencoder 对 MNIST 手写数字进行降噪和重建。

自编码器是**无监督/自监督表示学习**方法：不依赖人工类别标签，但训练时仍有清晰监督信号，即原始输入本身。

In [1]:
import math
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(23)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f" 当前设备：{device}")

 当前设备：mps


In [2]:
# ## 2. 自编码器的基本原理
# 最小的 shape 例子
x = torch.rand(4, 1, 28, 28)       # [batch=4, C=1, H=28, W=28]
z = torch.randn(4, 16)              # 每张图片压缩成 16 维表示
x_hat = torch.rand(4, 1, 28, 28)   # 重建结果必须同形状

loss = F.mse_loss(x_hat, x)
print(loss.shape)  # torch.Size([])，标量


torch.Size([])


In [3]:
## 3. 为什么“输入等于标签”不会让模型无意义地复制？
# 必须让 潜在向量z（16）的维度远小于输入向量x（28*28）的维度，这样 AE（Autoencoder）才有意义，这样才能在低维z向量中学到特征，否则就是简单的特征图复制了

# 下面的代码说明“没有瓶颈”与“有瓶颈”的维度差异：
input_dim = 28 * 28
x = torch.randn(8, input_dim) # (8, 28 * 28) 8表示批次，28*28表示数据输入维度，一般是图像经过展平后的结果

# 模拟未降维，图像复制
identy_like = nn.Sequential(
    nn.Linear(input_dim, input_dim),
    nn.ReLU(),
    nn.Linear(input_dim, input_dim)
)
print(identy_like(x).shape)        # torch.Size([8, 784]), 没发生特征降维，类似于图像复制

# 模拟维度压缩
bottleneck_encoder = nn.Sequential(
    nn.Linear(input_dim, 128),
    nn.ReLU(),
    nn.Linear(128, 16) # 特征压缩发生在这里 28*28 -> 16
)
print(bottleneck_encoder(x).shape) # torch.Size([8, 16]), 发生了特征降维，学习原始图像特征


# 模拟维度还原
z = bottleneck_encoder(x)
bottleneck_decoder = nn.Sequential(
    nn.Linear(16, 128),
    nn.ReLU(),
    nn.Linear(128, input_dim)
)
print(bottleneck_decoder(z).shape) # torch.Size([8, 784]), 维度还原


torch.Size([8, 784])
torch.Size([8, 16])
torch.Size([8, 784])


In [14]:
## 4. 编码器和解码器不是互为数学逆函数
# “编码”和“解码”描述其角色，不代表 `decoder(encoder(x))` 是严格可逆变换。由于存在降维、非线性和有限参数，信息通常会丢失，`x_hat` 只是对 `x` 的近似。


# encoder 编码器是将128维降为16维的
encoder = nn.Linear(128, 16)
# decoder 解码器是将16维升为128维的
decoder = nn.Linear(16, 128)

# x = torch.tensor([0.0, 1.1, 2.2, 3.3])
x = torch.randn(128)
print(x.shape)
z = encoder(x)
print(z.shape)
x_hat = decoder(z)
print(x_hat.shape)
print("reconstruction mse:", F.mse_loss(x_hat, x_hat).item())
print("reconstruction mse:", F.mse_loss(x_hat, x).item())
# 训练的目标不是保证每个任意输入都可逆，而是让模型在训练数据分布附近尽可能准确地 【【【重建】】】。

torch.Size([128])
torch.Size([16])
torch.Size([128])
reconstruction mse: 0.0
reconstruction mse: 1.1387320756912231


In [27]:
## 5. 选择重建损失：MSE、BCE 与输出激活

# | 输入/目标 | 常见输出层 | 常见损失 | 说明 |
# | --- | --- | --- | --- |
# | `[0, 1]` 连续灰度值 | `Sigmoid()` 或无激活 | `MSELoss` | 常见的图像重建起点 |
# | 二值像素或 Bernoulli 假设 | 不加 sigmoid | `BCEWithLogitsLoss` | 数值更稳定，内部含 sigmoid |
# | 已标准化为任意实数 | 无激活 | `MSELoss` / `L1Loss` | 不要再强加 sigmoid |
# | 对离群重建误差更稳健 | 无激活或按范围选 | `L1Loss` / SmoothL1 | 误差的绝对值更敏感度均匀 |

# x是连续值，使用sigmoid激活，使用MESLoss损失
# x是二值分类，不使用sigmoid，使用BEC损失
# x是离散的数据，不使用激活函数，使用L1Loss损失

# 使用 mes 的例子
mes_x = torch.randn(8, 16)
mse_target = torch.randint(0, 2, (8, 128)).float()
mse_encoder = nn.Sequential(
    nn.Linear(16, 128),
    nn.Sigmoid(),  # 将输出限制在 [0, 1]
)
mse_logist = mse_encoder(mes_x) # (8,128)
mse_loss = F.mse_loss(mse_logist, mse_target).item()
print(f"mse_logist.shape: {mse_logist.shape}")
print(f"mse_loss: {mse_loss}")


# 使用 BCE 版本
bce_encoder = nn.Linear(16, 128)
bce_x = torch.randn(8, 16)
bce_target = torch.randint(0, 2, (8, 128)).float()
bce_loss = nn.BCEWithLogitsLoss()
bce_logist = bce_encoder(bce_x)
bce_loss = bce_loss(bce_logist, bce_target).item()
print(f"bce_logist.shape: {bce_logist.shape}")
print(f"bce_loss: {bce_loss}")


mse_logist.shape: torch.Size([8, 128])
mse_loss: 0.2647784948348999
bce_logist.shape: torch.Size([8, 128])
bce_loss: 0.7447259426116943


In [ ]:
## 6. 最小可运行：二维数据上的线性自编码器
# 先不使用图片。我们生成接近直线 `y = 2x` 的二维点，再把 2 维输入压缩到 1 维。训练后，1 维潜在变量会倾向于表示点在线上的位置。
import matplotlib.pyplot as plt

n = 512
x1 = torch.linspace(-2, 2, n)
x2 = 2 * x1 + 0.15 * torch.randn(n)
points = torch.stack([x1, x2], dim=1)  # [512, 2] ，521个批次，每个批次的数据是2个维度

plt.scatter(points[:, 0], points[:, 1], s=8)
plt.axis("equal")
plt.show()

In [37]:
# 二维数据上的线性自编码器
class LinearAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 将 (x1, x2) 两个点的数据编码为一个数值，之后再将数值数据解码为两个点
        self.encoder = nn.Linear(2, 1)
        self.decoder = nn.Linear(1, 2)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

dataset = TensorDataset(points)
dataloader = DataLoader(points, batch_size=32, shuffle=True)

# 连续离散的点 不使用激活函数，使用L1Loss损失
model = LinearAutoEncoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    model.train()
    train_sample = 0
    train_loss = 0
    for points in dataloader:
        points = points.to(device)
        optimizer.zero_grad(set_to_none=True)
        x_hat, _ = model(points)
        loss = criterion(x_hat, points)
        loss.backward()
        optimizer.step()

        size = points.shape[0]
        train_sample += size
        train_loss += loss.item() * size

    loss = train_loss / train_sample
    if (epoch + 1) % 10 == 0:
        print(f"epoch={epoch + 1}, mse={loss:.6f}")

model.eval()
with torch.inference_mode():
    x_hat, z = model(points.to(device))

print(z.shape)      # [512, 1]
print(x_hat.shape)  # [512, 2]


epoch=10, mse=4.433821
epoch=20, mse=3.663674
epoch=30, mse=3.231970
epoch=40, mse=2.999362
epoch=50, mse=2.852063
epoch=60, mse=2.707355
epoch=70, mse=2.501110
epoch=80, mse=2.174932
epoch=90, mse=1.690468
epoch=100, mse=1.086079
epoch=110, mse=0.518890
epoch=120, mse=0.170575
epoch=130, mse=0.051516
epoch=140, mse=0.023995
epoch=150, mse=0.011979
epoch=160, mse=0.008186
epoch=170, mse=0.007801
epoch=180, mse=0.007199
epoch=190, mse=0.006692
epoch=200, mse=0.006548
torch.Size([32, 1])
torch.Size([32, 2])
